# 03c - RM-c: frozen encoder + Retrieval-Augmented Classification

RM-c tidak melatih apa pun. Ia memakai ulang dua artefak milik RM-b, yaitu
embedding beku dan head yang sudah terlatih, lalu menambahkan cabang retrieval.

Alur per sampel:

1. `p_bert` = softmax(head(embedding))
2. Cari k tetangga terdekat di indeks FAISS yang dibangun HANYA dari split train,
   lalu ubah label tetangga menjadi distribusi `p_retr`
3. `p_final = (1 - alpha) * p_bert + alpha * p_retr`, prediksi = argmax

Fusi dilakukan pada level probabilitas dan softmax hanya diterapkan sekali, di
cabang BERT sebelum fusi. Karena kedua masukan sudah berupa distribusi dan bobot
fusinya berjumlah satu, hasilnya sudah menjadi distribusi sah; softmax kedua
hanya akan meratakan selisih dan bisa mengubah argmax pada kasus nyaris seri.

Tuning RM-c adalah SATU grid: seluruh head RM-b (setiap head yang disimpan
`03b_rmb_frozen.ipynb`) x 66 konfigurasi `alpha x k` di
`tuning_grids/RMC_TUNING_GRID.csv`. Head dipilih lewat `rmb_run_id` di
konfigurasi RM-c, sehingga setiap kombinasi tercatat sebagai satu baris
`runs_rmc.csv` seperti run RM-a dan RM-b: resume, isolasi error, dan
`best.json` berlaku sama. Tidak ada training, indeks FAISS hanya dari train,
dan split test tidak dibuka.

Prasyarat: `03b_rmb_frozen.ipynb` sudah dijalankan (butuh seluruh state head di
`checkpoints/rmb_heads/`).


In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])


## 0. Pemulihan checkpoint

Checkpoint tidak ikut git (`outputs/**/checkpoints/` di-gitignore), sedangkan
`best.json` dan riwayat run ikut. Di clone atau instance baru `rmb_best.pt`
dan `rmb_heads/` tidak ada, dan menjalankan ulang 03b tidak membuatnya kembali:
F1 yang sama dengan juara tidak dipromosikan sehingga tidak disimpan. Sel di
bawah membangunnya dari konfigurasi yang tercatat di `runs_rmb.csv` tanpa
mengubah riwayat atau `best.json`. Bila checkpoint sudah ada, sel ini tidak
melakukan apa-apa.


In [ ]:
dipulihkan = runner.restore_checkpoints()
print(dipulihkan)


## 1. Head RM-b yang diuji

In [ ]:
riwayat_rmb = runner.reporter.runs_frame("rmb")
print(f"{len(riwayat_rmb)} head RM-b; indeks FAISS dari "
      f"{len(runner.features.labels['train'])} vektor train")
riwayat_rmb[["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
             "val_f1_macro", "trainable_params"]]


Indeks dibangun eksklusif dari split train. Kalau val atau test ikut masuk,
retrieval akan menemukan sampel uji di dalam indeksnya sendiri dan hasilnya
kehilangan makna.


## 2. Susun grid: seluruh head x alpha x k

Setiap head dipasangkan dengan 66 konfigurasi `alpha x k` yang sama
(`alpha` 0,0 sampai 1,0 step 0,1; `k` 1, 3, 5, 10, 20, 50; `weighting`
dikunci `similarity`). Head dan fusi digrid bersama karena keyakinan head
menentukan seberapa banyak cabang retrieval bisa membantu.

Seperti 03a dan 03b, `run_batch` melewati kombinasi yang sudah tercatat
(`resume=True`), jadi sel di bawah aman dijalankan ulang setelah terputus.


In [ ]:
import pandas as pd

GRID_DIR = settings.data_dir.parent / "tuning_grids"
grid_fusi = pd.read_csv(GRID_DIR / "RMC_TUNING_GRID.csv")

permintaan_rmc = [
    {"config": {"rmb_run_id": int(head),
                **{kunci: nilai for kunci, nilai in baris.items() if kunci != "catatan"}},
     "note": f"head RM-b #{int(head)}; {baris['catatan']}"}
    for head in riwayat_rmb["run_id"]
    for baris in grid_fusi.to_dict("records")
]
tersisa = runner.pending_requests("rmc", permintaan_rmc)
print(f"{len(grid_fusi)} konfigurasi alpha x k x {len(riwayat_rmb)} head = {len(permintaan_rmc)} run")
print(f"{len(permintaan_rmc) - len(tersisa)} selesai, {len(tersisa)} tersisa")


## 3. Jalankan grid

Tanpa training, tiap run hanya memuat head, membangun indeks FAISS, dan
menelusuri tetangga untuk split validation, sehingga seluruh grid selesai dalam
hitungan menit. Konfigurasi yang gagal diisolasi ke `runs_rmc_errors.csv`.


In [ ]:
runner.run_batch("rmc", permintaan_rmc, batch_id="rmc_grid_seluruh_head")

# Dibaca dari riwayat, bukan dari nilai kembalian run_batch, agar tetap jalan saat resume.
riwayat_rmc = runner.reporter.runs_frame("rmc")
grid_rmc = riwayat_rmc[riwayat_rmc["batch_id"] == "rmc_grid_seluruh_head"]
grid_rmc.nlargest(10, ["val_f1_macro", "val_f1_judi"])[
    ["run_id", "rmb_run_id", "head_arch_rmb", "hidden_dim_rmb", "alpha", "k",
     "val_f1_macro", "val_f1_judi", "val_f1_rmb", "head_trainable_params",
     "is_tie_with_best"]
]


`alpha=0` membuang cabang retrieval, jadi F1-nya harus sama persis dengan F1
head RM-b itu sendiri (`val_f1_rmb`). Ini pemeriksaan kewarasan bahwa head yang
dimuat adalah head yang dilatih di 03b.


In [ ]:
nol = grid_rmc[grid_rmc["alpha"] == 0.0]
selisih_pp = (nol["val_f1_macro"] - nol["val_f1_rmb"]).abs().max() * 100
print(f"alpha=0: selisih maksimum terhadap F1 head RM-b {selisih_pp:.4f} pp (harus 0)")


## 4. Pengaruh RAC per head

Kenaikan dari RAC diukur terhadap head itu sendiri (`alpha=0`). Tabel ini
menjawab apakah RAC membantu secara umum atau hanya di satu titik operasi, dan
apakah head yang lemah lebih terbantu daripada head yang kuat.


In [ ]:
ambang = settings.tie_threshold_pp
grid_rmc = grid_rmc.assign(gain_pp=(grid_rmc["val_f1_macro"] - grid_rmc["val_f1_rmb"]) * 100)

terbaik = grid_rmc.loc[grid_rmc.groupby("rmb_run_id")["val_f1_macro"].idxmax()]
per_head = terbaik[[
    "rmb_run_id", "head_arch_rmb", "hidden_dim_rmb", "head_trainable_params",
    "val_f1_rmb", "alpha", "k", "val_f1_macro", "gain_pp",
]].sort_values("val_f1_macro", ascending=False)

print(f"RAC menaikkan F1 di atas ambang seri {ambang} pp pada "
      f"{(per_head['gain_pp'] > ambang).sum()} dari {len(per_head)} head")
per_head.round(4)


In [ ]:
import matplotlib.pyplot as plt

fig, (kiri, kanan) = plt.subplots(1, 2, figsize=(12, 4.5))

kiri.scatter(per_head["val_f1_rmb"], per_head["gain_pp"])
kiri.axhline(ambang, color="gray", linestyle="--", label="ambang seri")
kiri.axhline(0, color="black", linewidth=0.8)
kiri.set_xlabel("F1-macro head RM-b sendiri (validation)")
kiri.set_ylabel("kenaikan terbaik dari RAC (pp)")
kiri.set_title("Kenaikan RAC vs kekuatan head")
kiri.legend(fontsize=8)
kiri.grid(alpha=0.3)

kurva = grid_rmc.groupby(["rmb_run_id", "alpha"])["gain_pp"].max().unstack("alpha")
for _, baris in kurva.iterrows():
    kanan.plot(kurva.columns, baris.values, color="tab:blue", alpha=0.25)
kanan.plot(kurva.columns, kurva.mean(), color="tab:red", linewidth=2.5, label="rata-rata head")
kanan.axhline(0, color="black", linewidth=0.8)
kanan.set_xlabel("alpha")
kanan.set_ylabel("kenaikan dari RAC (pp), k terbaik per alpha")
kanan.set_title("Kurva alpha per head")
kanan.legend(fontsize=8)
kanan.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(runner.reporter.figures_dir / "rmc_per_head.png", dpi=150)
plt.show()


## 5. Cek weighting=uniform di sel juara

`weighting` hanya berpengaruh bila `alpha > 0` dan relatif independen dari
`alpha x k`, jadi cukup dicoba sekali di sel juara grid (head, alpha, k yang
sama), seperti coordinate descent di 03a.


In [ ]:
import json

juara_grid = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))["rmc"]
runner.run_batch(
    "rmc",
    [{"config": {**juara_grid["config"], "weighting": "uniform"},
      "note": "tahap 2: weighting=uniform di sel juara grid seluruh head"}],
    batch_id="rmc_tahap2_weighting",
)

riwayat_rmc = runner.reporter.runs_frame("rmc")
riwayat_rmc[riwayat_rmc["batch_id"] == "rmc_tahap2_weighting"][
    ["run_id", "rmb_run_id", "alpha", "k", "weighting", "val_f1_macro",
     "delta_vs_best_f1_macro_pp", "is_tie_with_best"]
]


## 6. Juara RM-c

In [ ]:
juara = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))["rmc"]
print(f"run #{juara['run_id']} | val F1-macro {juara['val_f1_macro']:.4f}")
print(f"konfigurasi : {juara['config']}")
print(f"biaya head  : {juara['head_trainable_params']:,} parameter, "
      f"{juara['head_train_time_s']:.1f} s latih")


`best.json` memilih F1-macro tertinggi secara mekanis, sama seperti RM-a dan
RM-b. Kolom `is_tie_with_best` menandai kandidat yang selisihnya di bawah
ambang seri 0,15 pp; bila ada, pertimbangkan head yang lebih murah atau `k` yang
lebih kecil, dan catat alasannya di Bab 4.


## Ringkasan

RM-c tidak melatih apa pun, tetapi memakai head RM-b, sehingga biaya latihnya
adalah biaya head yang dipakai juara (`head_train_time_s` dan
`head_trainable_params` di `best.json`). `checkpoints/rmc_best.pt` membawa head
itu beserta konfigurasi fusinya. Biaya tambahannya ada di inferensi:
pembangunan indeks sekali dan penelusuran k tetangga per prediksi, diukur di
`05_final_benchmark.ipynb`.

Ketiga skenario sudah punya juara di `best.json`. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.
